# IFN680 Assessment 3 — Aircraft Classification (Development Notebook)

**Student:** Karan Rooprai  **Student ID:** N12498122

This notebook develops an image classifier for the 20-class FGVC-Aircraft subset using
**transfer learning with a pretrained ResNet-18**, following the techniques from the Week 4
tutorial and lecture.

The workflow is:

1. **Baseline** — ResNet-18 used as a fixed feature extractor (backbone frozen, only the new
   classification head is trained). This gives the performance *floor*.
2. **Three controlled experiments** — each changes **one** design choice relative to the
   baseline so we can isolate its effect:
   * **Experiment 1 — Data augmentation** (random flips / rotations / colour jitter).
   * **Experiment 2 — Fine-tuning** (unfreeze the backbone so all layers adapt to aircraft).
   * **Experiment 3 — AdamW optimiser + Cosine-Annealing learning-rate schedule.**
3. **Best model** — combine the winning ingredients, retrain on the **whole** `trainval` set,
   and save the weights for evaluation in `main_report.ipynb`.

All experiments are compared on a held-out **validation** split. The final test set is only
touched once, in `main_report.ipynb`, using the required metric **average per-class accuracy**.

> Run this notebook top-to-bottom on a GPU runtime. It saves `histories.pkl` (all learning
> curves + validation scores) and `best_model.pth` (the winning model's weights), which
> `main_report.ipynb` reloads to reproduce every figure and metric.

## 1. Setup

Standard imports, a fixed random seed for reproducibility, and automatic GPU selection —
exactly as in the Week 4 tutorial.

In [ ]:
import os
import copy
import random
import pickle

import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset

# Fix seeds so results are reproducible
SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Use the GPU if one is available
device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 2. Data

The dataset is an `ImageFolder` layout: `FGVCAircraft_Subset20/trainval` (1331 images, used for
training + validation) and `FGVCAircraft_Subset20/test` (669 images, used only for the final
evaluation). Each of the 20 sub-folders (`class_00 … class_19`) is one aircraft variant.

Jupyter sometimes leaves hidden `.ipynb_checkpoints` folders inside data directories, which
`ImageFolder` would mistake for an extra class and crash on. The first cell removes them.

In [ ]:
DATA_ROOT = 'FGVCAircraft_Subset20'

# Defensive cleanup: remove any hidden .ipynb_checkpoints folders that would break ImageFolder
for root, dirs, files in os.walk(DATA_ROOT):
    for d in list(dirs):
        if d == '.ipynb_checkpoints':
            import shutil
            shutil.rmtree(os.path.join(root, d))
            print("Removed", os.path.join(root, d))
print("Cleanup done.")

In [ ]:
# Load without transforms first; transforms are attached per-experiment below
trainval_dataset = torchvision.datasets.ImageFolder(os.path.join(DATA_ROOT, 'trainval'))
test_dataset     = torchvision.datasets.ImageFolder(os.path.join(DATA_ROOT, 'test'))

class_names = trainval_dataset.classes
num_classes = len(class_names)

print(f"Classes ({num_classes}): {class_names}")
print(f"Trainval images: {len(trainval_dataset)}")
print(f"Test images:     {len(test_dataset)}")

### 2.1 Stratified train / validation split

We hold out 20% of `trainval` as a validation set, stratified by class so every aircraft type
keeps the same proportion in both splits. The same `random_state` is reused everywhere, so all
experiments see identical train/val images and differences come only from the design change
under test.

In [ ]:
indices = list(range(len(trainval_dataset)))
labels  = [label for _, label in trainval_dataset.samples]

train_indices, val_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=labels,      # keep class balance in both splits
    random_state=42,
)
print(f"Train samples: {len(train_indices)}   Val samples: {len(val_indices)}")

### 2.2 Preprocessing and augmentation transforms

ResNet-18 was pretrained on ImageNet, so every image is resized to **224×224** and normalised
with the **ImageNet** mean/std. `eval_transform` is the plain preprocessing used for validation,
testing, and the non-augmented runs. `aug_transform` adds the Week-4 augmentations
(horizontal flip, small rotation, colour jitter) *before* the same preprocessing.

In [ ]:
imagenet_means = (0.485, 0.456, 0.406)
imagenet_stds  = (0.229, 0.224, 0.225)

# Plain preprocessing (no augmentation)
eval_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(imagenet_means, imagenet_stds),
])

# Preprocessing WITH augmentation (used only for training in the relevant experiments)
aug_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.ToTensor(),
    transforms.Resize((224, 224)),
    transforms.Normalize(imagenet_means, imagenet_stds),
])

### 2.3 DataLoader helper

`make_loaders` builds fresh train/val loaders for a given training transform. Because train and
val share the same underlying images, we `deepcopy` the dataset so augmentation is applied to the
training copy only — the validation copy always uses `eval_transform`. Setting
`full_trainval=True` returns a single loader over **all** of `trainval` (used to retrain the
final model).

In [ ]:
BATCH_SIZE = 32

def make_loaders(train_transform, full_trainval=False):
    """Return (train_loader, val_loader). If full_trainval, train on all trainval and val_loader is None."""
    base = torchvision.datasets.ImageFolder(os.path.join(DATA_ROOT, 'trainval'))

    if full_trainval:
        train_ds = copy.deepcopy(base)
        train_ds.transform = train_transform
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
        return train_loader, None

    train_ds = copy.deepcopy(base); train_ds.transform = train_transform
    val_ds   = copy.deepcopy(base); val_ds.transform   = eval_transform
    train_loader = DataLoader(Subset(train_ds, train_indices), batch_size=BATCH_SIZE, shuffle=True,  num_workers=2)
    val_loader   = DataLoader(Subset(val_ds,   val_indices),   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)
    return train_loader, val_loader

## 3. Model and training helpers

These reproduce the Week 4 tutorial helpers.

* `setup_model` swaps ResNet-18's 1000-way ImageNet head for a fresh 20-way head. If
  `freeze_backbone=True` every convolutional layer is frozen and only the new head trains
  (feature extraction); otherwise the whole network is trainable (fine-tuning).
* `train_epoch` / `eval_epoch` run one pass over the data and return the mean loss and accuracy.
* `run_training` drives the epoch loop, records the learning curves, and keeps the best model
  (by validation accuracy).

In [ ]:
def setup_model(model, num_classes, freeze_backbone=False):
    """Replace the final layer with a fresh num_classes head; optionally freeze the backbone."""
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
        for param in model.fc.parameters():   # keep the new head trainable
            param.requires_grad = True
    return model

In [ ]:
def train_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, targets in dataloader:
        inputs, targets = inputs.to(device), targets.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        predicted = torch.argmax(outputs, axis=1)
        correct += (predicted == targets).sum().item()
        total   += targets.size(0)
    return running_loss / total, correct / total


def eval_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in dataloader:
            inputs, targets = inputs.to(device), targets.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            running_loss += loss.item() * inputs.size(0)
            predicted = torch.argmax(outputs, axis=1)
            correct += (predicted == targets).sum().item()
            total   += targets.size(0)
    return running_loss / total, correct / total

In [ ]:
def run_training(model, train_loader, val_loader, optimizer, criterion,
                 epochs, scheduler=None, save_path=None):
    """Train for `epochs`, recording learning curves. Returns (history, best_val_acc).

    If a val_loader is given, the best model (by val accuracy) is kept and optionally saved.
    If val_loader is None (final retrain on full trainval), the last-epoch model is saved.
    """
    model = model.to(device)
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    best_val_acc, best_state = -1.0, None

    for epoch in range(epochs):
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer, device)
        history['train_loss'].append(tr_loss)
        history['train_acc'].append(tr_acc)

        if val_loader is not None:
            va_loss, va_acc = eval_epoch(model, val_loader, criterion, device)
            history['val_loss'].append(va_loss)
            history['val_acc'].append(va_acc)
            if va_acc > best_val_acc:
                best_val_acc = va_acc
                best_state = copy.deepcopy(model.state_dict())
            print(f"Epoch {epoch+1:2d}/{epochs} | "
                  f"train loss {tr_loss:.3f} acc {tr_acc:.3f} | "
                  f"val loss {va_loss:.3f} acc {va_acc:.3f}")
        else:
            print(f"Epoch {epoch+1:2d}/{epochs} | train loss {tr_loss:.3f} acc {tr_acc:.3f}")

        if scheduler is not None:
            scheduler.step()

    # For the final retrain (no val set) keep the last-epoch weights
    if best_state is None:
        best_state = copy.deepcopy(model.state_dict())
    if save_path is not None:
        torch.save(best_state, save_path)
        print(f"Saved weights -> {save_path}")

    history['best_val_acc'] = best_val_acc
    return history, best_val_acc

In [ ]:
# Collect every learning-curve history here so main_report.ipynb can reload and plot them
histories = {}
criterion = nn.CrossEntropyLoss()
EPOCHS = 12   # epochs per baseline/experiment run

## 4. Baseline — frozen ResNet-18 feature extractor

The reference point. ResNet-18 is loaded with ImageNet weights, the backbone is **frozen**, and
only the new 20-way head is trained with plain SGD (lr = 0.001, momentum = 0.9) on
**un-augmented** images. Its best validation accuracy is the *floor* every experiment must beat.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
baseline_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
optimizer = optim.SGD(filter(lambda p: p.requires_grad, baseline_model.parameters()),
                      lr=0.001, momentum=0.9)

histories['baseline'], _ = run_training(
    baseline_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 5. Experiment 1 — Data augmentation

**Hypothesis:** randomly flipping, rotating, and colour-jittering the training images exposes the
frozen feature extractor to more varied views of each aircraft, reducing over-fitting on the small
training set and improving validation accuracy.

**Controlled change vs baseline:** the *only* difference is the training transform
(`aug_transform` instead of `eval_transform`). Architecture, freezing, optimiser and epochs are
identical.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
aug_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(aug_transform)    # augmentation ON
optimizer = optim.SGD(filter(lambda p: p.requires_grad, aug_model.parameters()),
                      lr=0.001, momentum=0.9)

histories['aug'], _ = run_training(
    aug_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 6. Experiment 2 — Fine-tuning the backbone

**Hypothesis:** ImageNet features are generic; letting the whole network (not just the head) adapt
to aircraft should capture the fine-grained differences between variants and lift validation
accuracy above the frozen baseline.

**Controlled change vs baseline:** the *only* difference is that the backbone is **unfrozen**
(`freeze_backbone=False`), so every layer is trainable. The optimiser, learning rate (SGD,
lr = 0.001, momentum = 0.9), epochs, and (un-augmented) data are kept **identical** to the
baseline, so any change in accuracy is attributable to fine-tuning alone.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
finetune_model = setup_model(backbone, num_classes, freeze_backbone=False)  # all layers trainable

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
# Same optimiser + learning rate as the baseline -> the ONLY changed variable is freeze->unfreeze
optimizer = optim.SGD(finetune_model.parameters(), lr=0.001, momentum=0.9)

histories['finetune'], _ = run_training(
    finetune_model, train_loader, val_loader, optimizer, criterion, epochs=EPOCHS)

## 7. Experiment 3 — AdamW optimiser + Cosine-Annealing schedule

**Hypothesis:** replacing plain SGD with **AdamW** (adaptive updates + weight decay) and decaying
the learning rate along a **cosine** curve should give faster, more stable convergence of the
classification head and a small accuracy gain over the SGD baseline.

**Controlled change vs baseline:** only the optimiser and schedule change. The backbone stays
frozen and images stay un-augmented, so any difference is attributable to the optimisation
strategy.

In [ ]:
backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
adamw_model = setup_model(backbone, num_classes, freeze_backbone=True)

train_loader, val_loader = make_loaders(eval_transform)   # no augmentation
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, adamw_model.parameters()),
                        lr=0.001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

histories['adamw'], _ = run_training(
    adamw_model, train_loader, val_loader, optimizer, criterion,
    epochs=EPOCHS, scheduler=scheduler)

## 8. Experiment comparison

Best validation accuracy for the baseline and each experiment. The experiments that beat the
baseline are the ingredients we combine into the final model.

In [ ]:
print(f"{'Model':<28}{'Best val accuracy':>18}")
print('-' * 46)
for name, label in [('baseline', 'Baseline (frozen SGD)'),
                    ('aug',      'Exp 1: + Augmentation'),
                    ('finetune', 'Exp 2: Fine-tuning'),
                    ('adamw',    'Exp 3: AdamW + Cosine')]:
    print(f"{label:<28}{histories[name]['best_val_acc']:>18.4f}")

## 9. Best model — combine the winners, retrain on all `trainval`

The final model combines only the changes that **improved** validation accuracy over the baseline.
Fine-tuning (+0.25) and AdamW + Cosine-Annealing (+0.04) both helped, so both are kept.
Augmentation is **excluded**: it *reduced* validation accuracy for the frozen model (−0.10), so it
is not a winning ingredient and is dropped rather than carried into the final model.

The best model is therefore an **unfrozen** ResNet-18 trained with **AdamW + Cosine-Annealing** on
**un-augmented** images. Unlike the controlled experiments, this is a deliberately combined
configuration, so the learning rate is set to a smaller value (0.0001) suited to fine-tuning the
whole network with AdamW. As required, it is retrained on the **entire** `trainval` set (no
validation split) so it uses every labelled image before facing the test set. A slightly longer
schedule (15 epochs) is used, and the weights are saved to `best_model.pth`.

In [ ]:
FINAL_EPOCHS = 15

backbone = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.DEFAULT)
best_model = setup_model(backbone, num_classes, freeze_backbone=False)  # fine-tune (a winner)

# Winners only: fine-tuning + AdamW/cosine. Augmentation is dropped (it hurt the baseline),
# so the final model trains on un-augmented images (eval_transform).
full_train_loader, _ = make_loaders(eval_transform, full_trainval=True)  # all trainval, no augmentation
optimizer = optim.AdamW(best_model.parameters(), lr=0.0001, weight_decay=0.01)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=FINAL_EPOCHS)

histories['best'], _ = run_training(
    best_model, full_train_loader, None, optimizer, criterion,
    epochs=FINAL_EPOCHS, scheduler=scheduler, save_path='best_model.pth')

## 10. Save all learning curves

`histories.pkl` stores every learning curve and validation score. `main_report.ipynb` reloads it
(together with `best_model.pth`) to reproduce all figures, the comparison table, and the final
test metric without retraining.

In [ ]:
with open('histories.pkl', 'wb') as f:
    pickle.dump({'histories': histories, 'class_names': class_names}, f)
print("Saved histories.pkl")
print("Saved files:", [x for x in os.listdir('.') if x.endswith(('.pkl', '.pth'))])